# Pneumonia X-ray classification experiment

> **Research and education only.** This notebook is not a medical device and must not be used for diagnosis or care decisions.

Outputs were cleared after review because they contained stale machine-specific paths and large embedded figures. Exact metrics transcribed from the archived local run are preserved in `results/reported_metrics.json` and `results/archived_notebook_output.txt`. The dataset and trained weights are not included, so the run has not been independently reproduced from this repository.


In [ ]:
# CELL 1 — ENVIRONMENT CHECK
import tensorflow as tf
import numpy as np
import matplotlib
import sklearn
import seaborn
import PIL
import os

print("=" * 55)
print("        ENVIRONMENT CHECK")
print("=" * 55)
print(f"  TensorFlow   : {tf.__version__}")
print(f"  Keras        : {tf.keras.__version__}")
print(f"  Numpy        : {np.__version__}")
print(f"  Matplotlib   : {matplotlib.__version__}")
print(f"  Scikit-learn : {sklearn.__version__}")
print(f"  Pillow       : {PIL.__version__}")
print("=" * 55)

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    # Allow GPU memory growth — prevents crash on 8GB VRAM
    tf.config.experimental.set_memory_growth(gpus[0], True)
    print(f"  GPU DETECTED  : {gpus[0].name}")
    print(f"  Memory Growth : ENABLED (prevents OOM crash)")
else:
    print("  No GPU — training will use CPU (slower)")
print("=" * 55)

In [ ]:
# Portable dataset and output paths
from config import (
    CHARTS_DIR,
    CNN_PATH,
    DATASET_DIR,
    MOBILENET_PATH,
    MODELS_DIR,
    RESNET_PATH,
    TEST_DIR,
    TRAIN_DIR,
    VAL_DIR,
    ensure_output_dirs,
    validate_dataset_layout,
)

ensure_output_dirs()
validate_dataset_layout()

print(f"Dataset: {DATASET_DIR}")
print(f"Models: {MODELS_DIR}")
print(f"Charts: {CHARTS_DIR}")


In [ ]:
# Cell 3 - looking at some sample images from the dataset
# this helps us understand what the xray images actually look like
# before we start training any model

import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import random
import os

random.seed(42)

normal_folder    = os.path.join(TRAIN_DIR, "NORMAL")
pneumonia_folder = os.path.join(TRAIN_DIR, "PNEUMONIA")

normal_samples    = random.sample(os.listdir(normal_folder), 5)
pneumonia_samples = random.sample(os.listdir(pneumonia_folder), 5)

fig, axes = plt.subplots(2, 5, figsize=(16, 7))
fig.suptitle("Sample Chest Xray Images - Top: Normal  Bottom: Pneumonia", fontsize=13)

for col, img_name in enumerate(normal_samples):
    img = mpimg.imread(os.path.join(normal_folder, img_name))
    axes[0, col].imshow(img, cmap="gray")
    axes[0, col].set_title("Normal", color="green")
    axes[0, col].axis("off")

for col, img_name in enumerate(pneumonia_samples):
    img = mpimg.imread(os.path.join(pneumonia_folder, img_name))
    axes[1, col].imshow(img, cmap="gray")
    axes[1, col].set_title("Pneumonia", color="red")
    axes[1, col].axis("off")

plt.tight_layout()

save_path = os.path.join(CHARTS_DIR, "sample_images.png")
plt.savefig(save_path, dpi=150, bbox_inches="tight")
plt.show()

print("Sample images saved to:", save_path)

print()
print("Class imbalance in training set:")
print("Normal images    :", len(os.listdir(normal_folder)))
print("Pneumonia images :", len(os.listdir(pneumonia_folder)))

In [ ]:
# Cell 4 - setting up the data pipeline
# I load images in batches, apply augmentation to training data
# and compute class weights to handle the imbalance we saw in cell 3

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

# image size that ResNet50 and MobileNetV2 expect
IMG_SIZE   = (224, 224)
BATCH_SIZE = 32
SEED       = 42

# training data generator with augmentation
# augmentation means we slightly rotate, zoom, flip images
# this helps the model generalise better and not just memorise
train_datagen = ImageDataGenerator(
    rescale            = 1.0 / 255,
    rotation_range     = 10,
    zoom_range         = 0.1,
    horizontal_flip    = True,
    width_shift_range  = 0.1,
    height_shift_range = 0.1,
    shear_range        = 0.1,
    fill_mode          = "nearest",
    validation_split   = 0.2
)

# test data only gets rescaled, no augmentation
# I never augment test data as it must reflect real conditions
test_datagen = ImageDataGenerator(rescale = 1.0 / 255)

# 80 percent of train folder for actual training
train_gen = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size = IMG_SIZE,
    batch_size  = BATCH_SIZE,
    class_mode  = "binary",
    seed        = SEED,
    shuffle     = True,
    subset      = "training"
)

# 20 percent of train folder for validation during training
val_gen = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size = IMG_SIZE,
    batch_size  = BATCH_SIZE,
    class_mode  = "binary",
    seed        = SEED,
    shuffle     = False,
    subset      = "validation"
)

# completely separate test set - never touched during training
test_gen = test_datagen.flow_from_directory(
    TEST_DIR,
    target_size = IMG_SIZE,
    batch_size  = BATCH_SIZE,
    class_mode  = "binary",
    seed        = SEED,
    shuffle     = False
)

# compute class weights to handle the 3x imbalance
# this tells the model to penalise missing normal cases more
labels              = train_gen.classes
class_weights_array = compute_class_weight(
    class_weight = "balanced",
    classes      = np.unique(labels),
    y            = labels
)
class_weights = dict(enumerate(class_weights_array))

print("Data pipeline ready")
print()
print("Training images  :", train_gen.n)
print("Validation images:", val_gen.n)
print("Test images      :", test_gen.n)
print()
print("Class mapping    :", train_gen.class_indices)
print()
print("Class weights (to fix imbalance):")
print("  Normal (0)    :", round(class_weights[0], 4))
print("  Pneumonia (1) :", round(class_weights[1], 4))
print()

In [ ]:
# Cell 5 - building and training the ResNet50 model
# I use transfer learning here - ResNet50 is already trained on ImageNet
# I freeze the base first, train only my new layers, then fine tune

from tensorflow.keras.applications import ResNet50
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

# I load ResNet50 without its original top layer
# I set include_top=False so I can add my own classification head
resnet_base = ResNet50(weights="imagenet", include_top=False, input_shape=(224, 224, 3))

# I freeze all layers in phase 1 so I only train my new head
resnet_base.trainable = False

# I add my own layers on top of ResNet50
# GlobalAveragePooling reduces the feature maps to a single vector
# I then add Dense layers and Dropout to prevent overfitting
x = resnet_base.output
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation="relu")(x)
x = BatchNormalization()(x)
x = Dropout(0.5)(x)
x = Dense(128, activation="relu")(x)
x = Dropout(0.3)(x)
output = Dense(1, activation="sigmoid")(x)

# I create the full model connecting input to output
resnet_model = Model(inputs=resnet_base.input, outputs=output)

# I compile with a small learning rate since only the head is training
resnet_model.compile(
    optimizer = Adam(learning_rate=1e-4),
    loss      = "binary_crossentropy",
    metrics   = ["accuracy", "precision", "recall"]
)

print("ResNet50 model built")
print("Total parameters:", f"{resnet_model.count_params():,}")
print()

# I use three callbacks to control training
# EarlyStopping stops if validation loss stops improving
# ReduceLROnPlateau lowers learning rate when stuck
# ModelCheckpoint saves the best version of the model to disk
callbacks_resnet = [
    EarlyStopping(
        monitor              = "val_loss",
        patience             = 5,
        restore_best_weights = True,
        verbose              = 1
    ),
    ReduceLROnPlateau(
        monitor  = "val_loss",
        factor   = 0.3,
        patience = 3,
        min_lr   = 1e-7,
        verbose  = 1
    ),
    ModelCheckpoint(
        RESNET_PATH,
        monitor        = "val_loss",
        save_best_only = True,
        mode           = "min",
        verbose        = 1
    )
]

# phase 1 - I train only my classification head
# the ResNet base is frozen so its weights do not change
print("Phase 1 - training the classification head only")
print("ResNet base is frozen - only my added layers are training")
print()

history_resnet_p1 = resnet_model.fit(
    train_gen,
    epochs          = 20,
    validation_data = val_gen,
    class_weight    = class_weights,
    callbacks       = callbacks_resnet
)

# phase 2 - I unfreeze the last 15 layers of ResNet for fine tuning
# I use a much smaller learning rate to avoid destroying pretrained weights
print()
print("Phase 2 - fine tuning the last 15 ResNet layers")
print("I use a smaller learning rate of 1e-5 to fine tune carefully")
print()

for layer in resnet_base.layers[-15:]:
    layer.trainable = True

resnet_model.compile(
    optimizer = Adam(learning_rate=1e-5),
    loss      = "binary_crossentropy",
    metrics   = ["accuracy", "precision", "recall"]
)

history_resnet_p2 = resnet_model.fit(
    train_gen,
    epochs          = 10,
    validation_data = val_gen,
    class_weight    = class_weights,
    callbacks       = callbacks_resnet
)

# I load the best saved model from disk rather than the final epoch
# ModelCheckpoint saved the best version so I load that
print()
print("Loading best saved ResNet50 model from disk...")
resnet_best = load_model(RESNET_PATH)

# I evaluate on the test set which was never seen during training
test_gen.reset()
resnet_results = resnet_best.evaluate(test_gen, verbose=0)

print()
print("ResNet50 final results on test set:")
print("  Accuracy  :", round(resnet_results[1] * 100, 2), "%")
print("  Precision :", round(resnet_results[2] * 100, 2), "%")
print("  Recall    :", round(resnet_results[3] * 100, 2), "%")
print("  Loss      :", round(resnet_results[0], 4))
print()
print("ResNet50 model saved to:", RESNET_PATH)

# I combine both phase histories for plotting later in Cell 8
resnet_history_combined = {
    "accuracy"     : history_resnet_p1.history["accuracy"]     + history_resnet_p2.history["accuracy"],
    "val_accuracy" : history_resnet_p1.history["val_accuracy"] + history_resnet_p2.history["val_accuracy"],
    "loss"         : history_resnet_p1.history["loss"]         + history_resnet_p2.history["loss"],
    "val_loss"     : history_resnet_p1.history["val_loss"]     + history_resnet_p2.history["val_loss"],
}

print("Training history saved for plotting")
print()

In [ ]:
# Cell 6 - building and training the MobileNetV2 model
# I use transfer learning again but MobileNetV2 is much lighter than ResNet50
# it has fewer parameters so it trains faster and uses less memory

from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

# I reset the generators before starting a new model
train_gen.reset()
val_gen.reset()
test_gen.reset()

# I load MobileNetV2 without its original top layer
mobilenet_base = MobileNetV2(weights="imagenet", include_top=False, input_shape=(224, 224, 3))

# I freeze the base in phase 1 so only my head trains first
mobilenet_base.trainable = False

# I add my own classification head on top
# MobileNetV2 is smaller so I use slightly fewer neurons than ResNet50
x = mobilenet_base.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation="relu")(x)
x = BatchNormalization()(x)
x = Dropout(0.4)(x)
x = Dense(64, activation="relu")(x)
x = Dropout(0.3)(x)
output = Dense(1, activation="sigmoid")(x)

mobilenet_model = Model(inputs=mobilenet_base.input, outputs=output)

mobilenet_model.compile(
    optimizer = Adam(learning_rate=1e-4),
    loss      = "binary_crossentropy",
    metrics   = ["accuracy", "precision", "recall"]
)

print("MobileNetV2 model built")
print("Total parameters:", f"{mobilenet_model.count_params():,}")
print()

callbacks_mobile = [
    EarlyStopping(
        monitor              = "val_loss",
        patience             = 5,
        restore_best_weights = True,
        verbose              = 1
    ),
    ReduceLROnPlateau(
        monitor  = "val_loss",
        factor   = 0.3,
        patience = 3,
        min_lr   = 1e-7,
        verbose  = 1
    ),
    ModelCheckpoint(
        MOBILENET_PATH,
        monitor        = "val_loss",
        save_best_only = True,
        mode           = "min",
        verbose        = 1
    )
]

# phase 1 - I train only my classification head
print("Phase 1 - training the classification head only")
print("MobileNetV2 base is frozen")
print()

history_mobile_p1 = mobilenet_model.fit(
    train_gen,
    epochs          = 20,
    validation_data = val_gen,
    class_weight    = class_weights,
    callbacks       = callbacks_mobile
)

# phase 2 - I unfreeze the last 20 layers for fine tuning
# I use 1e-6 here not 1e-5 because MobileNetV2 depthwise layers break easily
print()
print("Phase 2 - fine tuning last 20 MobileNetV2 layers")
print("I use 1e-6 learning rate - MobileNetV2 needs a very gentle fine tune")
print()

for layer in mobilenet_base.layers[-20:]:
    layer.trainable = True

mobilenet_model.compile(
    optimizer = Adam(learning_rate=1e-6),
    loss      = "binary_crossentropy",
    metrics   = ["accuracy", "precision", "recall"]
)

history_mobile_p2 = mobilenet_model.fit(
    train_gen,
    epochs          = 10,
    validation_data = val_gen,
    class_weight    = class_weights,
    callbacks       = callbacks_mobile
)

# I load the best saved model from disk
print()
print("Loading best saved MobileNetV2 model from disk...")
mobilenet_best = load_model(MOBILENET_PATH)

test_gen.reset()
mobile_results = mobilenet_best.evaluate(test_gen, verbose=0)

print()
print("MobileNetV2 final results on test set:")
print("  Accuracy  :", round(mobile_results[1] * 100, 2), "%")
print("  Precision :", round(mobile_results[2] * 100, 2), "%")
print("  Recall    :", round(mobile_results[3] * 100, 2), "%")
print("  Loss      :", round(mobile_results[0], 4))
print()
print("MobileNetV2 model saved to:", MOBILENET_PATH)

# I save the combined history for plotting in Cell 8
mobilenet_history_combined = {
    "accuracy"     : history_mobile_p1.history["accuracy"]     + history_mobile_p2.history["accuracy"],
    "val_accuracy" : history_mobile_p1.history["val_accuracy"] + history_mobile_p2.history["val_accuracy"],
    "loss"         : history_mobile_p1.history["loss"]         + history_mobile_p2.history["loss"],
    "val_loss"     : history_mobile_p1.history["val_loss"]     + history_mobile_p2.history["val_loss"],
}

print("Training history saved for plotting")
print()

In [ ]:
# Cell 7 - building and training a custom CNN from scratch
# I build this entire network myself without using any pretrained weights
# I use 4 convolutional blocks to give the model enough depth to learn
# this is compared against ResNet50 and MobileNetV2 to show transfer learning is better

from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Conv2D, MaxPooling2D, BatchNormalization, Dense, Dropout, GlobalAveragePooling2D, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score

# I reset generators before starting
train_gen.reset()
val_gen.reset()
test_gen.reset()

# I lower the threshold to 0.35 to reduce the pneumonia bias
# without this the model tends to predict pneumonia for everything
CNN_THRESHOLD = 0.35

# I build the CNN completely from scratch with 4 convolutional blocks
# block 1 learns basic edges, block 4 learns complex xray patterns
cnn_model = Sequential([
    Input(shape=(224, 224, 3)),

    # block 1 - I detect basic edges and low level features
    Conv2D(32, (3, 3), activation="relu", padding="same"),
    BatchNormalization(),
    MaxPooling2D(2, 2),
    Dropout(0.25),

    # block 2 - I learn more detailed texture patterns
    Conv2D(64, (3, 3), activation="relu", padding="same"),
    BatchNormalization(),
    MaxPooling2D(2, 2),
    Dropout(0.25),

    # block 3 - I extract higher level features from the xray
    Conv2D(128, (3, 3), activation="relu", padding="same"),
    BatchNormalization(),
    MaxPooling2D(2, 2),
    Dropout(0.3),

    # block 4 - I learn the most complex patterns before classification
    Conv2D(128, (3, 3), activation="relu", padding="same"),
    BatchNormalization(),
    MaxPooling2D(2, 2),
    Dropout(0.3),

    # I convert the feature maps into a flat vector
    GlobalAveragePooling2D(),

    # classification head
    Dense(256, activation="relu"),
    Dropout(0.5),
    Dense(128, activation="relu"),
    Dropout(0.3),
    Dense(1, activation="sigmoid")
])

cnn_model.compile(
    optimizer = Adam(learning_rate=1e-4),
    loss      = "binary_crossentropy",
    metrics   = ["accuracy", "precision", "recall"]
)

# I use aggressive class weights because the CNN has no pretrained knowledge
# without boosting the normal class weight the model collapses
# I found through testing that normal weight of 5.0 prevents this
aggressive_weights = {
    0: 5.0,   # normal images - I heavily penalise missing these
    1: 0.5    # pneumonia images
}

print("Custom CNN built from scratch - 4 convolutional blocks")
print("Total parameters:", f"{cnn_model.count_params():,}")
print()
print("No pretrained weights - every weight starts random")
print("Class weights - Normal: 5.0   Pneumonia: 0.5")
print("Prediction threshold - 0.35")
print()

callbacks_cnn = [
    EarlyStopping(
        monitor              = "val_loss",
        patience             = 5,
        restore_best_weights = True,
        verbose              = 1
    ),
    ReduceLROnPlateau(
        monitor  = "val_loss",
        factor   = 0.3,
        patience = 3,
        min_lr   = 1e-7,
        verbose  = 1
    ),
    ModelCheckpoint(
        CNN_PATH,
        monitor        = "val_loss",
        save_best_only = True,
        mode           = "min",
        verbose        = 1
    )
]

print("Training started - maximum 15 epochs")
print()

history_cnn = cnn_model.fit(
    train_gen,
    epochs          = 15,
    validation_data = val_gen,
    class_weight    = aggressive_weights,
    callbacks       = callbacks_cnn
)

# I load the best saved checkpoint not the final epoch
print()
print("Loading best saved custom CNN from disk...")
cnn_best = load_model(CNN_PATH)

# I apply threshold 0.35 to get final predictions
test_gen.reset()
cnn_probs   = cnn_best.predict(test_gen, verbose=0)
true_labels = test_gen.classes
cnn_preds   = (cnn_probs > CNN_THRESHOLD).astype(int).flatten()

print()
print("Custom CNN final results on test set:")
print("  Accuracy  :", round(accuracy_score(true_labels, cnn_preds) * 100, 2), "%")
print("  Precision :", round(precision_score(true_labels, cnn_preds) * 100, 2), "%")
print("  Recall    :", round(recall_score(true_labels, cnn_preds) * 100, 2), "%")
print()
print("Custom CNN model saved to:", CNN_PATH)

# I save history for plotting later in Cell 8
cnn_history_combined = {
    "accuracy"     : history_cnn.history["accuracy"],
    "val_accuracy" : history_cnn.history["val_accuracy"],
    "loss"         : history_cnn.history["loss"],
    "val_loss"     : history_cnn.history["val_loss"],
}

print("Training history saved for plotting")
print()

In [ ]:
# Cell 8 - generating all comparison charts for the report
# I load all 3 saved models and evaluate them on the test set
# I then create 4 charts to visually compare their performance

import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import numpy as np
import os
warnings.filterwarnings("ignore")

from sklearn.metrics import confusion_matrix, roc_curve, auc, accuracy_score, precision_score, recall_score
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import load_model

# I reload all 3 best models from disk
print("Loading all 3 saved models from disk...")
resnet_model    = load_model(RESNET_PATH)
mobilenet_model = load_model(MOBILENET_PATH)
cnn_model       = load_model(CNN_PATH)
print("All 3 models loaded successfully")
print()

# I create a fresh test generator for evaluation
test_datagen_vis = ImageDataGenerator(rescale=1.0 / 255)
test_gen_vis = test_datagen_vis.flow_from_directory(
    TEST_DIR,
    target_size = (224, 224),
    batch_size  = 32,
    class_mode  = "binary",
    shuffle     = False
)
true_labels = test_gen_vis.classes
print("Test images loaded:", len(true_labels))
print()

# I get predictions from all 3 models
print("Getting predictions from all 3 models...")

test_gen_vis.reset()
resnet_probs = resnet_model.predict(test_gen_vis, verbose=0)
resnet_preds = (resnet_probs > 0.5).astype(int).flatten()

test_gen_vis.reset()
mobilenet_probs = mobilenet_model.predict(test_gen_vis, verbose=0)
mobilenet_preds = (mobilenet_probs > 0.5).astype(int).flatten()

test_gen_vis.reset()
cnn_probs_raw = cnn_model.predict(test_gen_vis, verbose=0)
cnn_preds     = (cnn_probs_raw > 0.35).astype(int).flatten()

print("Predictions done for all 3 models")
print()

# chart 1 - confusion matrices for all 3 models
# I use these to show where each model makes mistakes
print("Creating chart 1 - confusion matrices...")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Confusion Matrices - All Three Models", fontsize=14)

models_data = [
    ("ResNet50",    resnet_preds,    "#2196F3"),
    ("MobileNetV2", mobilenet_preds, "#4CAF50"),
    ("Custom CNN",  cnn_preds,       "#FF5722"),
]

for ax, (name, preds, color) in zip(axes, models_data):
    cm     = confusion_matrix(true_labels, preds)
    cm_pct = cm.astype("float") / cm.sum(axis=1)[:, np.newaxis] * 100
    sns.heatmap(cm_pct, annot=True, fmt=".1f",
                cmap=sns.light_palette(color, as_cmap=True),
                ax=ax,
                xticklabels=["Normal", "Pneumonia"],
                yticklabels=["Normal", "Pneumonia"],
                cbar=False,
                annot_kws={"size": 13})
    for i in range(2):
        for j in range(2):
            ax.text(j + 0.5, i + 0.72, f"({cm[i,j]})",
                    ha="center", fontsize=9, color="gray")
    tn, fp, fn, tp = cm.ravel()
    acc = (tp + tn) / (tp + tn + fp + fn) * 100
    rec = tp / (tp + fn) * 100
    ax.set_title(f"{name}\nAccuracy: {acc:.1f}%   Recall: {rec:.1f}%", fontsize=11)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True Label")

plt.tight_layout()
path = os.path.join(CHARTS_DIR, "confusion_matrices.png")
plt.savefig(path, dpi=150, bbox_inches="tight")
plt.show()
print("Saved:", path)
print()

# chart 2 - roc curves for all 3 models
# I use these to show how well each model separates the two classes
print("Creating chart 2 - ROC curves...")

fig, ax = plt.subplots(figsize=(8, 7))

roc_data = [
    ("ResNet50",    resnet_probs.flatten(),    "#2196F3"),
    ("MobileNetV2", mobilenet_probs.flatten(), "#4CAF50"),
    ("Custom CNN",  cnn_probs_raw.flatten(),   "#FF5722"),
]

for name, probs, color in roc_data:
    fpr, tpr, _ = roc_curve(true_labels, probs)
    roc_auc     = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=color, lw=2.5, label=f"{name}  AUC = {roc_auc:.3f}")

ax.plot([0, 1], [0, 1], color="gray", lw=1.5, linestyle="--", label="Random  AUC = 0.500")
ax.set_xlabel("False Positive Rate", fontsize=12)
ax.set_ylabel("True Positive Rate", fontsize=12)
ax.set_title("ROC Curves - All Three Models", fontsize=13)
ax.legend(loc="lower right", fontsize=11)
ax.grid(alpha=0.3)

plt.tight_layout()
path = os.path.join(CHARTS_DIR, "roc_curves.png")
plt.savefig(path, dpi=150, bbox_inches="tight")
plt.show()
print("Saved:", path)
print()

# chart 3 - bar chart comparing accuracy precision and recall
# I use this to give a clear side by side view of all 3 models
print("Creating chart 3 - metric comparison bar chart...")

model_names = ["ResNet50", "MobileNetV2", "Custom CNN"]
all_preds   = [resnet_preds, mobilenet_preds, cnn_preds]
acc_scores  = [accuracy_score(true_labels, p) * 100  for p in all_preds]
prec_scores = [precision_score(true_labels, p) * 100 for p in all_preds]
rec_scores  = [recall_score(true_labels, p) * 100    for p in all_preds]

x     = np.arange(len(model_names))
width = 0.25

fig, ax = plt.subplots(figsize=(11, 6))
b1 = ax.bar(x - width, acc_scores,  width, label="Accuracy",  color="#1565C0", alpha=0.85)
b2 = ax.bar(x,          prec_scores, width, label="Precision", color="#2E7D32", alpha=0.85)
b3 = ax.bar(x + width,  rec_scores,  width, label="Recall",    color="#BF360C", alpha=0.85)

for bars in [b1, b2, b3]:
    for bar in bars:
        h = bar.get_height()
        ax.annotate(f"{h:.1f}%",
                    xy=(bar.get_x() + bar.get_width() / 2, h),
                    xytext=(0, 4), textcoords="offset points",
                    ha="center", fontsize=9)

ax.set_xlabel("Model", fontsize=12)
ax.set_ylabel("Score (%)", fontsize=12)
ax.set_title("Model Performance Comparison - Accuracy, Precision, Recall", fontsize=13)
ax.set_xticks(x)
ax.set_xticklabels(model_names, fontsize=12)
ax.set_ylim([0, 115])
ax.legend(fontsize=11)
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
path = os.path.join(CHARTS_DIR, "metric_comparison.png")
plt.savefig(path, dpi=150, bbox_inches="tight")
plt.show()
print("Saved:", path)
print()

# chart 4 - training history showing accuracy and loss per epoch
# I use this to show how each model learned over time
print("Creating chart 4 - training history...")

histories = {
    "ResNet50"   : resnet_history_combined,
    "MobileNetV2": mobilenet_history_combined,
    "Custom CNN" : cnn_history_combined,
}
colors_h = {
    "ResNet50"   : "#2196F3",
    "MobileNetV2": "#4CAF50",
    "Custom CNN" : "#FF5722",
}

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("Training History - Accuracy and Loss per Epoch", fontsize=13)

for name, hist in histories.items():
    c = colors_h[name]
    axes[0].plot(hist["accuracy"],     color=c, lw=2,               label=f"{name} train")
    axes[0].plot(hist["val_accuracy"], color=c, lw=2, linestyle="--", label=f"{name} val")
    axes[1].plot(hist["loss"],         color=c, lw=2,               label=f"{name} train")
    axes[1].plot(hist["val_loss"],     color=c, lw=2, linestyle="--", label=f"{name} val")

axes[0].set_title("Accuracy over Epochs")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Accuracy")
axes[0].legend(fontsize=7)
axes[0].grid(alpha=0.3)

axes[1].set_title("Loss over Epochs")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].legend(fontsize=7)
axes[1].grid(alpha=0.3)

plt.tight_layout()
path = os.path.join(CHARTS_DIR, "training_history.png")
plt.savefig(path, dpi=150, bbox_inches="tight")
plt.show()
print("Saved:", path)
print()

# final summary
print("All 4 charts saved to:", CHARTS_DIR)
print()
print("Final results summary:")
print()
for name, acc, prec, rec in zip(model_names, acc_scores, prec_scores, rec_scores):
    tag = "  <-- recommended" if name == "MobileNetV2" else ""
    print(f"  {name:<15}  Accuracy: {acc:.2f}%   Precision: {prec:.2f}%   Recall: {rec:.2f}%{tag}")
print()

In [ ]:
# Cell 9 - live inference demo using the best model MobileNetV2
# I load the saved model and run it on real test images
# I show the prediction, model score and model-score band for each image
# this is what I will demonstrate live during my presentation

import numpy as np
import matplotlib.pyplot as plt
import os
import random
import warnings
warnings.filterwarnings("ignore")

from PIL import Image
import tensorflow as tf

# I load only MobileNetV2 for the demo as it is the best performing model
print("Loading MobileNetV2 for live demo...")
demo_model = tf.keras.models.load_model(MOBILENET_PATH)
print("Model loaded successfully")
print()

# I pick 3 normal and 3 pneumonia images from the test set
random.seed(42)
demo_images = []

for folder, label in [("NORMAL", 0), ("PNEUMONIA", 1)]:
    path  = os.path.join(TEST_DIR, folder)
    files = [f for f in os.listdir(path)
             if f.lower().endswith((".jpg", ".jpeg", ".png"))]
    picks = random.sample(files, 3)
    for f in picks:
        demo_images.append((os.path.join(path, f), label))

random.shuffle(demo_images)

class_names = ["NORMAL", "PNEUMONIA"]

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle("Live Inference Demo - MobileNetV2\nGreen border = Correct   Red border = Wrong",
             fontsize=13)

correct = 0
results = []

print("Running predictions on 6 test images...")
print()

for idx, (img_path, true_label) in enumerate(demo_images[:6]):
    ax = axes[idx // 3][idx % 3]

    # I open and preprocess the image exactly like during training
    img = Image.open(img_path).convert("RGB").resize((224, 224))
    arr = np.array(img) / 255.0

    # I get the prediction probability from the model
    prob       = demo_model.predict(np.expand_dims(arr, 0), verbose=0)[0][0]
    pred_label = 1 if prob > 0.5 else 0
    conf       = prob if pred_label == 1 else 1 - prob
    ok         = pred_label == true_label
    if ok:
        correct += 1

    # I assign a model-score band based on the probability score
    if prob >= 0.85:
        score_band = "HIGH MODEL SCORE"
    elif prob >= 0.65:
        score_band = "MEDIUM MODEL SCORE"
    else:
        score_band = "LOW MODEL SCORE"

    color = "#2E7D32" if ok else "#C62828"
    results.append((class_names[true_label], class_names[pred_label], conf * 100, score_band, ok))

    # I display the image with a coloured border showing correct or wrong
    ax.imshow(arr, cmap="gray")
    ax.axis("off")
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_edgecolor(color)
        spine.set_linewidth(5)

    ax.set_title(
        f"True: {class_names[true_label]}\n"
        f"Pred: {class_names[pred_label]} ({conf*100:.1f}%)\n"
        f"Score band: {score_band}",
        fontsize=9,
        color=color
    )

plt.tight_layout()
path = os.path.join(CHARTS_DIR, "live_inference_demo.png")
plt.savefig(path, dpi=150, bbox_inches="tight")
plt.show()

# I print a clean results table
print("Live inference results:")
print()
for true, pred, conf, score_band, ok in results:
    sym = "CORRECT" if ok else "WRONG"
    print(f"  True: {true:<12}  Predicted: {pred:<12}  Model score: {conf:.1f}%  Score band: {score_band:<18}  {sym}")

print()
print(f"Score: {correct} out of 6 correct ({correct/6*100:.0f}%)")
print()
print("Demo chart saved to:", path)
print()

In [ ]:
import os, tensorflow as tf, matplotlib.pyplot as plt
from tensorflow.keras import Sequential
from tensorflow.keras.layers import (Conv2D, BatchNormalization, MaxPooling2D,
                                     Dropout, GlobalAveragePooling2D, Dense)

OUT_DIR = str(CHARTS_DIR)

# ── FIGURE 6 — Class Distribution Bar Chart ───────────────
categories = ['Normal\n(Train)', 'Pneumonia\n(Train)', 'Normal\n(Test)', 'Pneumonia\n(Test)']
counts     = [1341, 3875, 234, 390]
colors     = ['#2E7D32', '#C62828', '#66BB6A', '#EF5350']

fig, ax = plt.subplots(figsize=(9, 6))
bars = ax.bar(categories, counts, color=colors, width=0.5, edgecolor='black')
for bar, count in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 40,
            str(count), ha='center', fontsize=12, fontweight='bold')

ax.set_title('Class Distribution — Chest X-Ray Dataset\n(Kermany et al., 2018)',
             fontsize=14, fontweight='bold')
ax.set_ylabel('Number of Images', fontsize=12)
ax.set_ylim(0, 4400)
ax.axhline(y=1341, color='#2E7D32', linestyle='--', alpha=0.4, label='Normal train count')
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'class_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved: class_distribution.png")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import os

OUT_DIR = str(CHARTS_DIR)

layers = [
    ("Input\n224 × 224 × 3",              "#4A90D9", "(None, 224, 224, 3)"),
    ("Conv2D — 32 filters, 3×3, ReLU",    "#E67E22", "(None, 224, 224, 32)"),
    ("BatchNormalization",                  "#27AE60", "(None, 224, 224, 32)"),
    ("MaxPooling2D — 2×2",                 "#1ABC9C", "(None, 112, 112, 32)"),
    ("Dropout — 0.25",                     "#95A5A6", "(None, 112, 112, 32)"),
    ("Conv2D — 64 filters, 3×3, ReLU",    "#E67E22", "(None, 112, 112, 64)"),
    ("BatchNormalization",                  "#27AE60", "(None, 112, 112, 64)"),
    ("MaxPooling2D — 2×2",                 "#1ABC9C", "(None,  56,  56, 64)"),
    ("Dropout — 0.25",                     "#95A5A6", "(None,  56,  56, 64)"),
    ("Conv2D — 128 filters, 3×3, ReLU",   "#E67E22", "(None,  56,  56, 128)"),
    ("BatchNormalization",                  "#27AE60", "(None,  56,  56, 128)"),
    ("MaxPooling2D — 2×2",                 "#1ABC9C", "(None,  28,  28, 128)"),
    ("Dropout — 0.25",                     "#95A5A6", "(None,  28,  28, 128)"),
    ("Conv2D — 128 filters, 3×3, ReLU",   "#E67E22", "(None,  28,  28, 128)"),
    ("BatchNormalization",                  "#27AE60", "(None,  28,  28, 128)"),
    ("MaxPooling2D — 2×2",                 "#1ABC9C", "(None,  14,  14, 128)"),
    ("Dropout — 0.25",                     "#95A5A6", "(None,  14,  14, 128)"),
    ("GlobalAveragePooling2D",             "#8E44AD", "(None, 128)"),
    ("Dense — 256 units, ReLU",            "#C0392B", "(None, 256)"),
    ("Dropout — 0.50",                     "#95A5A6", "(None, 256)"),
    ("Dense — 128 units, ReLU",            "#C0392B", "(None, 128)"),
    ("Dropout — 0.30",                     "#95A5A6", "(None, 128)"),
    ("Output — Dense 1, Sigmoid",          "#2C3E50", "(None, 1)"),
]

n        = len(layers)
fig_h    = n * 1.05 + 2
fig, ax  = plt.subplots(figsize=(12, fig_h))
ax.set_xlim(0, 10)
ax.set_ylim(-1, n * 1.05 + 0.5)
ax.axis('off')

box_w = 7.0
box_h = 0.72
x0    = 1.5

for i, (label, color, shape) in enumerate(reversed(layers)):
    y = i * 1.05
    rect = mpatches.FancyBboxPatch(
        (x0, y), box_w, box_h,
        boxstyle="round,pad=0.05",
        linewidth=1.5,
        edgecolor='white',
        facecolor=color,
        alpha=0.93,
        zorder=2
    )
    ax.add_patch(rect)

    # Layer name
    ax.text(x0 + 0.3, y + box_h / 2, label,
            ha='left', va='center',
            fontsize=10, fontweight='bold', color='white', zorder=3)

    # Shape on right
    ax.text(x0 + box_w - 0.2, y + box_h / 2, shape,
            ha='right', va='center',
            fontsize=9, color='white',
            fontstyle='italic', zorder=3)

    # Arrow
    if i < n - 1:
        ax.annotate("",
            xy=(x0 + box_w/2, y + box_h + 0.005),
            xytext=(x0 + box_w/2, y + box_h + 0.33),
            arrowprops=dict(arrowstyle="<-", color="#333333", lw=1.8),
            zorder=1
        )

# Legend
legend = [
    mpatches.Patch(color="#4A90D9", label="Input"),
    mpatches.Patch(color="#E67E22", label="Conv2D"),
    mpatches.Patch(color="#27AE60", label="BatchNorm"),
    mpatches.Patch(color="#1ABC9C", label="MaxPooling"),
    mpatches.Patch(color="#95A5A6", label="Dropout"),
    mpatches.Patch(color="#8E44AD", label="GlobalAvgPool"),
    mpatches.Patch(color="#C0392B", label="Dense"),
    mpatches.Patch(color="#2C3E50", label="Output"),
]
ax.legend(handles=legend, loc='upper center',
          ncol=4, fontsize=9,
          bbox_to_anchor=(0.5, 1.01),
          framealpha=0.9)

ax.set_title("Custom CNN Architecture — Pneumonia Detection",
             fontsize=14, fontweight='bold', y=1.025)

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'cnn_architecture.png'),
            dpi=200, bbox_inches='tight', facecolor='white')
plt.show()
print("✅ Saved: cnn_architecture.png")

In [ ]:
# Cell 7B - architecture diagrams for all 3 models (WHITE BACKGROUND)
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ─────────────────────────────────────────
# DIAGRAM 1 — ResNet50
# ─────────────────────────────────────────
layers_resnet = [
    ("Input",              "224 x 224 x 3 Image",                                           "#2E86C1"),
    ("ResNet50 Base",      "Pretrained ImageNet | 24M+ params\nPhase1: Frozen  |  Phase2: Last 15 layers unfrozen", "#E67E22"),
    ("GlobalAvgPool2D",    "Flatten feature maps into 1D vector",                            "#7D3C98"),
    ("Dense(256)",         "ReLU Activation",                                                "#1E8449"),
    ("BatchNormalization", "Normalize layer outputs",                                        "#717D7E"),
    ("Dropout(0.5)",       "50% neurons dropped during training",                            "#C0392B"),
    ("Dense(128)",         "ReLU Activation",                                                "#1E8449"),
    ("Dropout(0.3)",       "30% neurons dropped during training",                            "#C0392B"),
    ("Dense(1) + Sigmoid", "Output:  Normal (0)  /  Pneumonia (1)",                         "#E67E22"),
]

fig, ax = plt.subplots(figsize=(10, 14))
fig.patch.set_facecolor("white")
ax.set_facecolor("white")
ax.set_xlim(0, 10)
ax.set_ylim(0, len(layers_resnet) * 1.6 + 1)
ax.axis("off")

box_w, box_h, x0, gap = 7.5, 1.0, 1.25, 1.55

for i, (name, desc, color) in enumerate(layers_resnet):
    y = (len(layers_resnet) - 1 - i) * gap + 1.0
    if i < len(layers_resnet) - 1:
        y_below = (len(layers_resnet) - 2 - i) * gap + 1.0
        ax.annotate("", xy=(5, y_below + box_h + 0.05), xytext=(5, y - 0.05),
                    arrowprops=dict(arrowstyle="->", color="#333333", lw=2.0))
    rect = mpatches.FancyBboxPatch((x0, y), box_w, box_h,
        boxstyle="round,pad=0.1", linewidth=1.8,
        edgecolor="#cccccc", facecolor=color, alpha=0.90)
    ax.add_patch(rect)
    ax.text(x0 + box_w/2, y + box_h*0.68, name,
            ha="center", va="center", fontsize=12, fontweight="bold",
            color="white", fontfamily="monospace")
    ax.text(x0 + box_w/2, y + box_h*0.28, desc,
            ha="center", va="center", fontsize=8.5, color="white", style="italic")

fig.text(0.5, 0.98, "ResNet50 Transfer Learning Architecture",
         ha="center", fontsize=14, fontweight="bold", color="#1a1a1a")
fig.text(0.5, 0.965, "Binary Pneumonia Detection  |  Custom Classification Head",
         ha="center", fontsize=10, color="#555555")

plt.tight_layout()
save_path = os.path.join(CHARTS_DIR, "architecture_resnet50.png")
plt.savefig(save_path, dpi=150, bbox_inches="tight", facecolor="white")
plt.show()
print("Saved:", save_path)


# ─────────────────────────────────────────
# DIAGRAM 2 — MobileNetV2
# ─────────────────────────────────────────
layers_mobile = [
    ("Input",              "224 x 224 x 3 Image",                                           "#2E86C1"),
    ("MobileNetV2 Base",   "Pretrained ImageNet | 2.4M params\nPhase1: Frozen  |  Phase2: Last 20 layers unfrozen", "#E67E22"),
    ("GlobalAvgPool2D",    "Flatten feature maps into 1D vector",                            "#7D3C98"),
    ("Dense(128)",         "ReLU Activation",                                                "#1E8449"),
    ("BatchNormalization", "Normalize layer outputs",                                        "#717D7E"),
    ("Dropout(0.4)",       "40% neurons dropped during training",                            "#C0392B"),
    ("Dense(64)",          "ReLU Activation",                                                "#1E8449"),
    ("Dropout(0.3)",       "30% neurons dropped during training",                            "#C0392B"),
    ("Dense(1) + Sigmoid", "Output:  Normal (0)  /  Pneumonia (1)",                         "#E67E22"),
]

fig, ax = plt.subplots(figsize=(10, 14))
fig.patch.set_facecolor("white")
ax.set_facecolor("white")
ax.set_xlim(0, 10)
ax.set_ylim(0, len(layers_mobile) * 1.6 + 1)
ax.axis("off")

for i, (name, desc, color) in enumerate(layers_mobile):
    y = (len(layers_mobile) - 1 - i) * gap + 1.0
    if i < len(layers_mobile) - 1:
        y_below = (len(layers_mobile) - 2 - i) * gap + 1.0
        ax.annotate("", xy=(5, y_below + box_h + 0.05), xytext=(5, y - 0.05),
                    arrowprops=dict(arrowstyle="->", color="#333333", lw=2.0))
    rect = mpatches.FancyBboxPatch((x0, y), box_w, box_h,
        boxstyle="round,pad=0.1", linewidth=1.8,
        edgecolor="#cccccc", facecolor=color, alpha=0.90)
    ax.add_patch(rect)
    ax.text(x0 + box_w/2, y + box_h*0.68, name,
            ha="center", va="center", fontsize=12, fontweight="bold",
            color="white", fontfamily="monospace")
    ax.text(x0 + box_w/2, y + box_h*0.28, desc,
            ha="center", va="center", fontsize=8.5, color="white", style="italic")

fig.text(0.5, 0.98, "MobileNetV2 Transfer Learning Architecture",
         ha="center", fontsize=14, fontweight="bold", color="#1a1a1a")
fig.text(0.5, 0.965, "Binary Pneumonia Detection  |  Custom Classification Head",
         ha="center", fontsize=10, color="#555555")

plt.tight_layout()
save_path = os.path.join(CHARTS_DIR, "architecture_mobilenetv2.png")
plt.savefig(save_path, dpi=150, bbox_inches="tight", facecolor="white")
plt.show()
print("Saved:", save_path)

print()
print("Both architecture diagrams saved to:", CHARTS_DIR)